# 03 · Async and streaming

**Estimated study time: 12 minutes.**

You already know programming from Java. Here we focus on what feels different in Python. Run the cells in order, predict the results, and change one value before moving on.

## Coroutines vs concurrent execution

A call to `async def` produces a coroutine, not its final value. `await` lets other tasks use the event loop while this task waits for I/O. `asyncio.gather()` runs awaitables concurrently; it does **not** create CPU-parallel threads.

In [ ]:
import asyncio
import time

async def fake_api(name: str, delay: float = 0.05) -> str:
    await asyncio.sleep(delay)
    return name.upper()

start = time.perf_counter()
sequential = [await fake_api("a"), await fake_api("b"), await fake_api("c")]
sequential_time = time.perf_counter() - start
start = time.perf_counter()
concurrent = await asyncio.gather(fake_api("a"), fake_api("b"), fake_api("c"))
concurrent_time = time.perf_counter() - start
assert sequential == list(concurrent)
print("Sequential", round(sequential_time, 2), "Concurrent", round(concurrent_time, 2))

## Generators for streaming responses

`yield` produces items lazily. A generator does not compute its whole output up front. AI token streams often have this shape.

In [ ]:
def stream_words(message: str):
    for word in message.split():
        yield word

assert list(stream_words("response ready now")) == ["response", "ready", "now"]
for chunk in stream_words("first second third"):
    print(chunk)

## Failures and timeouts

Real SDKs can time out or fail. Handle failures at a useful boundary and do not silently discard them.

In [ ]:
async def unreliable_call() -> str:
    await asyncio.sleep(0.02)
    return "ok"

try:
    result = await asyncio.wait_for(unreliable_call(), timeout=0.5)
except asyncio.TimeoutError:
    result = "timeout"
print(result)

### Exercise

Modify the three calls to use three different delays. Which result comes first in the returned list? `gather` preserves input order even when completion order differs.